## Day 5: Data Cleaning and Exploratory Analysis

This section cleans the employee dataset and explores salary, experience, departments, cities, and attrition.


### Task 1: Data Cleaning

The CSV had 21 rows, including one exact duplicate. After removing that duplicate, 20 employee records remained. Salary had one missing value, which I filled with the median salary of 64,000. Experience had one missing value; I left it unknown and flagged it instead of guessing.

I trimmed extra spaces in text columns and converted ID, age, salary, and experience to numeric types. I used the IQR rule to flag possible salary outliers and kept flagged records for review because an unusual salary is not automatically an error.


### Before and After Cleaning

| Measure | Before | After |
|---|---:|---:|
| Rows | 21 | 20 |
| Missing values | 2 | 1 |
| Duplicate rows | 1 | 0 |

The one remaining missing value is the employee’s unknown experience. The cleaned dataset was saved as `clean_data.csv`.


### Task 2: Exploratory Analysis

I calculated summary statistics for salary, age, and experience, compared employee counts and average salaries by department, and examined relationships between numeric columns. I also compared attrition rates and median salaries across groups.


### Business Questions and Answers

1. **Which department has the most employees?** IT has the most, with 5 employees.
2. **Which department has the highest average salary?** IT has the highest average salary, 80,800.
3. **What is the attrition rate in each department?** Marketing: 66.7%; HR: 25%; Sales: 25%; IT: 20%; Finance: 0%.
4. **What is the median salary by city?** Bangalore: 66,000; Chennai: 60,750; Delhi: 49,000; Hyderabad: 61,750; Mumbai: 70,500.
5. **Which department has the highest attrition rate?** Marketing, at 66.7% (2 of 3 employees).


### Three Key Findings

- IT is the largest department, with 5 of the 20 employees.
- IT has the highest average salary at 80,800. HR has the lowest average salary at 51,500.
- Marketing has the highest attrition rate: 2 of its 3 employees, or 66.7%. This is a small group, so the percentage should be treated cautiously.


### Outlier Review, Recommendations, and Limitations

The IQR rule identifies salaries that are unusually far from the middle of the salary distribution. A flag does not prove a salary is wrong, so flagged values were kept for review.

**Recommendations:** Review salary levels by role and seniority, and investigate the reasons for Marketing’s attrition with more employee records and feedback.

**Limitations:** This is a small practice dataset, so its patterns may not represent a real workforce. The missing salary was filled with the median, and the missing experience value remains unknown. The department budget values used in the merge example are fictional practice values, not actual budgets. Correlation does not prove causation.





In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("employees.csv")

rows_before = len(df)
missing_before = int(df.isna().sum().sum())
duplicates_before = int(df.duplicated().sum())

print("Original shape:", df.shape)
print("Original missing values:", missing_before)
print("Original duplicate rows:", duplicates_before)

Original shape: (21, 8)
Original missing values: 2
Original duplicate rows: 1


In [3]:
# Remove exact duplicate rows
df = df.drop_duplicates().copy()

# Clean salary text, such as "78,000", and convert it to a number
df["salary"] = (
    df["salary"]
    .astype("string")
    .str.replace(",", "", regex=False)
)
df["salary"] = pd.to_numeric(df["salary"], errors="coerce")

# Convert other numeric columns
for column in ["id", "age", "experience_years"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Trim spaces from text columns
for column in ["name", "department", "city", "attrition"]:
    df[column] = df[column].astype("string").str.strip()

# Fill missing salary with its median
salary_median = df["salary"].median()
df["salary"] = df["salary"].fillna(salary_median)

# Leave unknown experience blank, but flag it
df["experience_years_missing"] = df["experience_years"].isna()

# Find possible salary outliers using the IQR rule; keep them and flag them
q1 = df["salary"].quantile(0.25)
q3 = df["salary"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df["salary_outlier_iqr"] = (
    (df["salary"] < lower_bound) |
    (df["salary"] > upper_bound)
)

cleaning_summary = pd.DataFrame({
    "measure": ["Rows", "Missing values", "Duplicate rows"],
    "before": [
        rows_before,
        missing_before,
        duplicates_before
    ],
    "after": [
        len(df),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum())
    ]
})

print("Cleaning summary:")
display(cleaning_summary)

# Save the cleaned data and summary as CSV files
df.to_csv("clean_data.csv", index=False)
cleaning_summary.to_csv("cleaning_summary.csv", index=False)

print("Saved clean_data.csv and cleaning_summary.csv")

Cleaning summary:


,measure,before,after
0,Rows,21,20
1,Missing values,2,1
2,Duplicate rows,1,0


Saved clean_data.csv and cleaning_summary.csv


In [4]:
numeric_columns = ["salary", "age", "experience_years"]

statistics = df[numeric_columns].agg(
    ["mean", "median", "min", "max", "std"]
).T

print("Numeric summary statistics:")
display(statistics)

department_summary = df.groupby("department").agg(
    employee_count=("id", "count"),
    average_salary=("salary", "mean"),
    median_salary=("salary", "median"),
    average_experience=("experience_years", "mean")
).sort_values("average_salary", ascending=False)

print("Department summary:")
display(department_summary)

Numeric summary statistics:


,mean,median,min,max,std
salary,64450.0,64000.0,43000.0,92000.0,13674.698111
age,33.1,32.5,25.0,41.0,4.644181
experience_years,8.473684,8.0,2.0,16.0,3.892563


Department summary:


,employee_count,average_salary,median_salary,average_experience
department,,,,
IT,5,80800.0,81000.0,12.200000
Finance,4,71000.0,70500.0,10.666667
Marketing,3,59333.333333,59000.0,7.666667
Sales,4,54250.0,55000.0,6.500000
HR,4,51500.0,49500.0,4.750000


In [5]:
numeric_columns = ["salary", "age", "experience_years"]

statistics = df[numeric_columns].agg(
    ["mean", "median", "min", "max", "std"]
).T

print("Numeric summary statistics:")
display(statistics)

department_summary = df.groupby("department").agg(
    employee_count=("id", "count"),
    average_salary=("salary", "mean"),
    median_salary=("salary", "median"),
    average_experience=("experience_years", "mean")
).sort_values("average_salary", ascending=False)

print("Department summary:")
display(department_summary)

Numeric summary statistics:


,mean,median,min,max,std
salary,64450.0,64000.0,43000.0,92000.0,13674.698111
age,33.1,32.5,25.0,41.0,4.644181
experience_years,8.473684,8.0,2.0,16.0,3.892563


Department summary:


,employee_count,average_salary,median_salary,average_experience
department,,,,
IT,5,80800.0,81000.0,12.200000
Finance,4,71000.0,70500.0,10.666667
Marketing,3,59333.333333,59000.0,7.666667
Sales,4,54250.0,55000.0,6.500000
HR,4,51500.0,49500.0,4.750000


In [6]:
correlations = df[numeric_columns].corr()

upper_triangle = correlations.where(
    np.triu(np.ones(correlations.shape), k=1).astype(bool)
)

top_correlations = (
    upper_triangle.stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={
        "level_0": "variable_1",
        "level_1": "variable_2"
    })
)

top_correlations["absolute_correlation"] = (
    top_correlations["correlation"].abs()
)

top_correlations = top_correlations.sort_values(
    "absolute_correlation",
    ascending=False
).head(3)

print("Top numeric correlations:")
display(top_correlations)

Top numeric correlations:


,variable_1,variable_2,correlation,absolute_correlation
5,age,experience_years,0.997320,0.997320
2,salary,experience_years,0.924330,0.924330
1,salary,age,0.895124,0.895124


In [7]:
department_counts = df["department"].value_counts()
highest_count_department = department_counts.idxmax()

highest_salary_department = department_summary["average_salary"].idxmax()

attrition_rate = (
    df["attrition"].str.casefold().eq("yes")
    .groupby(df["department"])
    .mean()
    .mul(100)
)

city_median_salary = df.groupby("city")["salary"].median()
highest_attrition_department = attrition_rate.idxmax()

print(
    f"1. Most employees: {highest_count_department}, "
    f"{department_counts.max()} employees."
)

print(
    f"2. Highest average salary: {highest_salary_department}, "
    f"{department_summary.loc[highest_salary_department, 'average_salary']:.2f}."
)

print("3. Attrition rate by department:")
display(attrition_rate.round(1).rename("attrition_percent").to_frame())

print("4. Median salary by city:")
display(city_median_salary.rename("median_salary").to_frame())

print(
    f"5. Highest attrition rate: {highest_attrition_department}, "
    f"{attrition_rate.loc[highest_attrition_department]:.1f}%."
)

1. Most employees: IT, 5 employees.
2. Highest average salary: IT, 80800.00.
3. Attrition rate by department:


,attrition_percent
department,
Finance,0.0
HR,25.0
IT,20.0
Marketing,66.7
Sales,25.0


4. Median salary by city:


,median_salary
city,
Bangalore,66000.0
Chennai,61500.0
Delhi,49000.0
Hyderabad,69500.0
Mumbai,70500.0


5. Highest attrition rate: Marketing, 66.7%.


In [8]:
budget_table = pd.DataFrame({
    "department": ["Sales", "IT", "HR", "Finance", "Marketing"],
    "monthly_budget_inr": [200000, 300000, 150000, 180000, 160000]
})

department_with_budget = pd.merge(
    department_summary.reset_index(),
    budget_table,
    on="department",
    how="left"
)

print("Department summary with example budgets:")
display(department_with_budget)

attrition_pivot = pd.pivot_table(
    df,
    index="department",
    columns="attrition",
    values="id",
    aggfunc="count",
    fill_value=0
)

print("Department by attrition:")
display(attrition_pivot)

Department summary with example budgets:


,department,employee_count,average_salary,median_salary,average_experience,monthly_budget_inr
0,IT,5,80800.0,81000.0,12.200000,300000
1,Finance,4,71000.0,70500.0,10.666667,180000
2,Marketing,3,59333.333333,59000.0,7.666667,160000
3,Sales,4,54250.0,55000.0,6.500000,200000
4,HR,4,51500.0,49500.0,4.750000,150000


Department by attrition:


attrition,No,Yes
department,,
Finance,4,0
HR,3,1
IT,4,1
Marketing,1,2
Sales,3,1
